In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ==============================
# CONFIGURATION (UPDATED)
# ==============================
RAW_IMG_DIR = r"E:\new_dataset\non_labeled"
RAW_MASK_DIR = r"E:\new_dataset\labeled"

# 🔥 NEW OUTPUT FOLDER (do NOT overwrite old one)
OUT_DIR = r"E:\processed_segmentation_dataset_224"

IMG_SIZE = 224   # 🔥 Reduced from 256 → faster training
SPLITS = ["train", "val", "test"]
CLASSES = ["fake", "real"]
MASK_EXTENSIONS = [".png", ".jpg", ".jpeg"]

# ==============================
# CREATE OUTPUT DIRECTORIES
# ==============================
for split in SPLITS:
    os.makedirs(os.path.join(OUT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, split, "masks"), exist_ok=True)

# ==============================
# RED MASK → BINARY MASK
# ==============================
def red_mask_to_binary(mask_bgr):
    """
    Red pixels -> 255
    Background -> 0
    """
    red_channel = mask_bgr[:, :, 2]
    binary = (red_channel > 120).astype(np.uint8) * 255
    return binary

# ==============================
# PREPROCESSING
# ==============================
total_saved = 0

for split in SPLITS:
    print(f"\n🔄 Processing {split} split")

    for cls in CLASSES:
        img_dir = os.path.join(RAW_IMG_DIR, split, cls)
        mask_dir = os.path.join(RAW_MASK_DIR, split, cls)

        if not os.path.isdir(img_dir):
            continue

        for fname in tqdm(os.listdir(img_dir), desc=f"{split}/{cls}"):

            img_path = os.path.join(img_dir, fname)
            name, _ = os.path.splitext(fname)

            # ---------- FIND MATCHING MASK ----------
            mask_path = None
            for ext in MASK_EXTENSIONS:
                candidate = os.path.join(mask_dir, name + ext)
                if os.path.exists(candidate):
                    mask_path = candidate
                    break

            if mask_path is None:
                continue

            # ---------- LOAD ----------
            image = cv2.imread(img_path)
            mask_bgr = cv2.imread(mask_path)

            if image is None or mask_bgr is None:
                continue

            # ---------- RESIZE (IMPORTANT FIX) ----------
            image = cv2.resize(
                image,
                (IMG_SIZE, IMG_SIZE),
                interpolation=cv2.INTER_LINEAR
            )

            mask_bgr = cv2.resize(
                mask_bgr,
                (IMG_SIZE, IMG_SIZE),
                interpolation=cv2.INTER_NEAREST  # 🔥 critical for masks
            )

            # ---------- MASK CONVERSION ----------
            binary_mask = red_mask_to_binary(mask_bgr)

            # ---------- SAFE OUTPUT NAME ----------
            out_name = f"{cls}_{name}.png"

            # ---------- SAVE ----------
            cv2.imwrite(
                os.path.join(OUT_DIR, split, "images", out_name),
                image
            )
            cv2.imwrite(
                os.path.join(OUT_DIR, split, "masks", out_name),
                binary_mask
            )

            total_saved += 1

print("\n✅ Preprocessing completed successfully")
print(f"📁 Total image-mask pairs saved: {total_saved}")
print(f"📂 Output directory: {OUT_DIR}")



🔄 Processing train split


train/real: 100%|████████████████████████████████████████████████████████████| 115814/115814 [2:17:53<00:00, 14.00it/s]



🔄 Processing val split


val/real: 100%|████████████████████████████████████████████████████████████████████| 1656/1656 [00:29<00:00, 55.99it/s]



🔄 Processing test split


test/real: 100%|███████████████████████████████████████████████████████████████████| 1901/1901 [00:32<00:00, 57.74it/s]


✅ Preprocessing completed successfully
📁 Total image-mask pairs saved: 340468
📂 Output directory: E:\processed_segmentation_dataset_224
